# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install duckdb pandas numpy


In [3]:
!pip install duckdb pandas numpy


# ============================================================
# SETUP
# ============================================================

import os
import duckdb
import pandas as pd
import numpy as np

from getpass import getpass


# ------------------------------------------------------------
# Hugging Face connection
# ------------------------------------------------------------

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    hf_token = getpass(
        "Enter your Hugging Face READ token: "
    )

con = duckdb.connect()

con.execute(
    "SET enable_progress_bar = false"
)

con.execute(
    "SET VARIABLE hf_token = ?",
    [hf_token]
)

con.execute("""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")


# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

REL = (
    "hf://datasets/FlyRank/internship-warehouse"
)

FACT = (
    f"{REL}/fact_content_daily_performance"
)

DIM_CONTENT = (
    f"{REL}/dim_content.parquet"
)

DIM_CLIENTS = (
    f"{REL}/dim_clients.parquet"
)


# ------------------------------------------------------------
# Prediction setup
# ------------------------------------------------------------

# The application is daily.
#
# We do NOT require a fixed amount of history.
# Each page uses all valid historical observations available
# before the scoring date.
#
# This notebook uses the available 2026 data to demonstrate
# the baseline queue.

START_DATE = "2026-01-01"

END_DATE = "2026-06-30"


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUTPUT_DIR = "work/outputs"

OUTPUT_PATH = (
    f"{OUTPUT_DIR}/baseline_action_score.csv"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


print("Setup complete.")
print("Historical data starts:", START_DATE)
print("Historical data ends:", END_DATE)
print("Prediction unit: next-day page search position")

Enter your Hugging Face READ token: ··········
Setup complete.
Historical data starts: 2026-01-01
Historical data ends: 2026-06-30
Prediction unit: next-day page search position


# My rule and its reason codes

The application forecasts the next search location of every page.

I only use observations prior to the scoring date in this baseline.

No specific requirement for how the five months are accomplished. A page can make a contribution
The moment it has a good enough number of valid historical observations to make a directional
baseline.

Correct interpretation of search position:

- a lower position number corresponds to a better position number,
A larger position number will be a worse rank.

The baseline first attempts the next position based on recent observed
position behaviour.

The action score is then given more weight to pages that recently and
The predicted position means deteriorating.

Supporting signals such as impressions, CTR and observed activity are
The context is used instead of the target to be predicted.

It is a clear score for decision support, a baseline score.

It is NOT a probability, it is NOT the final trained ML prediction.

Reason codes:
- PREDICTED_RANK_DEGRADATION — next position is predicted as degradation.
- RECENT_RANK_DEGRADATION: recent observed position is deteriorating.
- LOW_CTR — CTR recently has been weak.
LOW_VISIBILITY — recent impressions are relatively low.
- INSUFFICIENT_EVIDENCE — Not enough observations useful for reaching a conclusion.
- NO_MAJOR_SIGNAL: No major negative signal found.

In [4]:
# ============================================================
# SECTION 1 — RULE AND REASON CODES
# ============================================================

REASON_CODES = [
    "RANK_DETERIORATION",
    "PERSISTENT_RANK_DETERIORATION",
    "WEAK_POSITION",
    "RANK_VOLATILITY",
    "LOW_CTR",
    "DECLINING_VISIBILITY",
    "INSUFFICIENT_EVIDENCE",
    "NO_MAJOR_SIGNAL"
]

ACTION_LEVELS = [
    "HIGH_ATTENTION",
    "REVIEW",
    "MONITOR",
    "INSUFFICIENT_EVIDENCE"
]

print("Reason codes:")

for code in REASON_CODES:
    print("-", code)

print("\nAction levels:")

for action in ACTION_LEVELS:
    print("-", action)

Reason codes:
- RANK_DETERIORATION
- PERSISTENT_RANK_DETERIORATION
- WEAK_POSITION
- RANK_VOLATILITY
- LOW_CTR
- DECLINING_VISIBILITY
- INSUFFICIENT_EVIDENCE
- NO_MAJOR_SIGNAL

Action levels:
- HIGH_ATTENTION
- REVIEW
- MONITOR
- INSUFFICIENT_EVIDENCE


## 2. Build the ranked queue (writes the CSV)

The latest historical information is used to build the queue, page by page.

For every page:

1. Find its latest observed day.
2. Only make observations prior to the next prediction day.
3. Make an estimate of the next position based on past position behaviour.
4. Compare the estimated position to the most recent observed position.
5. Prioritize pages to be given greater importance of deteriorating ranks.
6. Provide CTR, impressions and activity as supporting evidence.

The length of the history available may vary from page to page.

If not exactly five, then at least five, pages in the navigation bar.
months of history.

This score is used to rank the pages from highest to lowest priority.
lowest priority.

In [ ]:
# ============================================================
# SECTION 2 — BUILD THE RANKED QUEUE
# ============================================================

import numpy as np
import pandas as pd
import time

print("Starting page-level ranking queue...", flush=True)


# ------------------------------------------------------------
# STEP 1 — Read data
# ------------------------------------------------------------

print("Step 1/4: Reading daily page data...", flush=True)

start = time.time()

months = pd.date_range(
    START_DATE,
    END_DATE,
    freq="MS"
).strftime("%Y-%m").tolist()

paths = ",".join(
    f"'{FACT}/month={month}/*.parquet'"
    for month in months
)

query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    AVG(gsc_avg_position) AS avg_position,

    SUM(COALESCE(gsc_impressions, 0)) AS impressions,

    SUM(COALESCE(gsc_clicks, 0)) AS clicks

FROM read_parquet(
    [{paths}],
    union_by_name = true
)

WHERE gsc_avg_position IS NOT NULL
  AND gsc_avg_position > 0

GROUP BY
    report_date,
    client_hash_id,
    content_hash_id

ORDER BY
    client_hash_id,
    content_hash_id,
    report_date
"""

df = con.execute(query).df()

print(
    f"Data loaded: {len(df):,} rows "
    f"({time.time() - start:.1f}s)",
    flush=True
)


# ------------------------------------------------------------
# STEP 2 — Prepare data
# ------------------------------------------------------------

print(
    "Step 2/4: Preparing page observations...",
    flush=True
)

df["report_date"] = pd.to_datetime(
    df["report_date"]
)

df["avg_position"] = pd.to_numeric(
    df["avg_position"],
    errors="coerce"
)

df["impressions"] = pd.to_numeric(
    df["impressions"],
    errors="coerce"
).fillna(0)

df["clicks"] = pd.to_numeric(
    df["clicks"],
    errors="coerce"
).fillna(0)

df = df[
    df["avg_position"].notna()
    &
    (df["avg_position"] > 0)
].copy()

print(
    f"Found {df['content_hash_id'].nunique():,} pages",
    flush=True
)


# ------------------------------------------------------------
# STEP 3 — Build page-level features
# ------------------------------------------------------------

print(
    "Step 3/4: Calculating page-level features...",
    flush=True
)

rows = []

groups = df.groupby(
    ["client_hash_id", "content_hash_id"],
    sort=False
)

total_pages = len(groups)

for i, ((client, page), g) in enumerate(
    groups,
    start=1
):

    g = (
        g
        .sort_values("report_date")
        .reset_index(drop=True)
    )

    # Need at least two observations
    if len(g) < 2:
        continue


    # --------------------------------------------------------
    # Position history
    # --------------------------------------------------------

    position = g["avg_position"].astype(float)

    recent = position.tail(7)

    latest = position.iloc[-1]

    previous = position.iloc[-2]

    latest_date = g["report_date"].iloc[-1]


    # --------------------------------------------------------
    # Position trend
    #
    # Positive slope = position number increasing
    #                = rank getting worse
    #
    # Negative slope = position number decreasing
    #                = rank improving
    # --------------------------------------------------------

    if len(recent) >= 2:

        x = np.arange(len(recent))

        slope = np.polyfit(
            x,
            recent.to_numpy(),
            1
        )[0]

    else:

        slope = np.nan


    # --------------------------------------------------------
    # Recent position change
    # --------------------------------------------------------

    if previous != 0:

        position_change = (
            latest - previous
        ) / abs(previous)

    else:

        position_change = np.nan


    # --------------------------------------------------------
    # Baseline next-day prediction
    # --------------------------------------------------------

    if pd.notna(slope):

        predicted_position = (
            0.60 * latest
            +
            0.40 * (
                latest + slope
            )
        )

    else:

        predicted_position = latest


    # --------------------------------------------------------
    # Predicted change
    #
    # Positive = expected rank deterioration
    # Negative = expected rank improvement
    # --------------------------------------------------------

    predicted_change = (
        predicted_position - latest
    )


    if latest != 0:

        predicted_change_pct = (
            predicted_change
            /
            abs(latest)
        ) * 100

    else:

        predicted_change_pct = np.nan


    # --------------------------------------------------------
    # Recent observations
    # --------------------------------------------------------

    changes = position.diff()

    worsening_days = int(
        (changes > 0).sum()
    )

    improving_days = int(
        (changes < 0).sum()
    )


    # --------------------------------------------------------
    # Latest CTR
    # --------------------------------------------------------

    latest_impressions = (
        g["impressions"].iloc[-1]
    )

    latest_clicks = (
        g["clicks"].iloc[-1]
    )

    if latest_impressions > 0:

        latest_ctr = (
            latest_clicks
            /
            latest_impressions
        )

    else:

        latest_ctr = np.nan


    # --------------------------------------------------------
    # Recent CTR
    # --------------------------------------------------------

    recent_impressions = (
        g["impressions"]
        .tail(7)
        .sum()
    )

    recent_clicks = (
        g["clicks"]
        .tail(7)
        .sum()
    )

    if recent_impressions > 0:

        recent_ctr = (
            recent_clicks
            /
            recent_impressions
        )

    else:

        recent_ctr = np.nan


    # --------------------------------------------------------
    # Add page
    # --------------------------------------------------------

    rows.append({

        "client_hash_id":
            client,

        "content_hash_id":
            page,

        "latest_date":
            latest_date,

        "prediction_date":
            latest_date + pd.Timedelta(days=1),

        "history_days":
            len(g),

        "latest_position":
            latest,

        "previous_position":
            previous,

        "recent_position_mean":
            recent.mean(),

        "historical_position_mean":
            position.mean(),

        "recent_position_std":
            recent.std(),

        "recent_position_change":
            position_change,

        "position_slope":
            slope,

        "baseline_predicted_position":
            predicted_position,

        "predicted_position_change":
            predicted_change,

        "predicted_position_change_pct":
            predicted_change_pct,

        "latest_impressions":
            latest_impressions,

        "recent_impressions":
            g["impressions"].tail(7).mean(),

        "latest_ctr":
            latest_ctr,

        "recent_ctr":
            recent_ctr,

        "worsening_days":
            worsening_days,

        "improving_days":
            improving_days
    })


    # Progress
    if (
        i % 500 == 0
        or i == total_pages
    ):

        print(
            f"Processed "
            f"{i:,}/{total_pages:,} pages...",
            flush=True
        )


# ------------------------------------------------------------
# STEP 4 — Create final queue
# ------------------------------------------------------------

print(
    "Step 4/4: Creating final ranking queue...",
    flush=True
)

queue_df = pd.DataFrame(rows)


if queue_df.empty:

    raise ValueError(
        "No eligible pages were found."
    )


# ------------------------------------------------------------
# Rank initially by predicted deterioration
# ------------------------------------------------------------

queue_df = queue_df.sort_values(
    "predicted_position_change",
    ascending=False
).reset_index(drop=True)


print()
print("COMPLETE")
print("=" * 50)
print(
    f"Pages in queue : {len(queue_df):,}"
)
print(
    f"Rows in source : {len(df):,}"
)
print("=" * 50)

display(
    queue_df.head(10)
)

Starting page-level ranking queue...
Step 1/4: Reading daily page data...


## 3. Top-20 review

The Top-20 list is a review queue, not a claim that these pages will
definitely lose rank.

For each page I show:

- current observed position,
- baseline next-position estimate,
- predicted position change,
- action level,
- reason code,
- confidence note,
- what could make the baseline wrong.

A higher priority means that the transparent baseline found stronger
evidence of possible rank deterioration.

The baseline does not provide a probability of ranking decline.

In [11]:
# ============================================================
# SECTION 3 — TOP-20 REVIEW
# ============================================================

TOP_N = 20


# ------------------------------------------------------------
# Start from the page-level queue
# ------------------------------------------------------------

output_df = queue_df.copy()


# ------------------------------------------------------------
# Action score
#
# This is a directional decision-support score.
# It is NOT a probability.
# ------------------------------------------------------------

deterioration = (
    output_df["predicted_position_change"]
    .clip(lower=0)
    .fillna(0)
)

trend_signal = (
    output_df["position_slope"]
    .clip(lower=0)
    .fillna(0)
)

recent_signal = (
    output_df["recent_position_change"]
    .clip(lower=0)
    .fillna(0)
)


# ------------------------------------------------------------
# Percentile ranks
# ------------------------------------------------------------

deterioration_rank = (
    deterioration
    .rank(
        pct=True,
        method="average"
    )
    .fillna(0)
)

trend_rank = (
    trend_signal
    .rank(
        pct=True,
        method="average"
    )
    .fillna(0)
)

recent_rank = (
    recent_signal
    .rank(
        pct=True,
        method="average"
    )
    .fillna(0)
)


# ------------------------------------------------------------
# Final action score
# ------------------------------------------------------------

output_df["action_score"] = (
    0.60 * deterioration_rank
    +
    0.25 * trend_rank
    +
    0.15 * recent_rank
) * 100


# ------------------------------------------------------------
# Action level
# ------------------------------------------------------------

output_df["action_level"] = "MONITOR"


output_df.loc[
    output_df["history_days"] < 3,
    "action_level"
] = "INSUFFICIENT_EVIDENCE"


output_df.loc[
    (
        output_df["action_score"] >= 75
    )
    &
    (
        output_df["predicted_position_change"] > 0
    )
    &
    (
        output_df["history_days"] >= 3
    ),
    "action_level"
] = "HIGH_ATTENTION"


output_df.loc[
    (
        output_df["action_score"] >= 50
    )
    &
    (
        output_df["action_score"] < 75
    )
    &
    (
        output_df["predicted_position_change"] > 0
    )
    &
    (
        output_df["history_days"] >= 3
    ),
    "action_level"
] = "REVIEW"


# ------------------------------------------------------------
# Reason code
# ------------------------------------------------------------

output_df["reason_code"] = "NO_MAJOR_SIGNAL"


output_df.loc[
    output_df["predicted_position_change"] > 0,
    "reason_code"
] = "PREDICTED_RANK_DEGRADATION"


output_df.loc[
    (
        output_df["predicted_position_change"] > 0
    )
    &
    (
        output_df["recent_position_change"] > 0
    ),
    "reason_code"
] = "PERSISTENT_RANK_DETERIORATION"


# ------------------------------------------------------------
# Weak CTR
# ------------------------------------------------------------

ctr_threshold = (
    output_df["recent_ctr"]
    .median()
)

output_df.loc[
    (
        output_df["recent_ctr"] < ctr_threshold
    )
    &
    (
        output_df["predicted_position_change"] > 0
    )
    &
    (
        output_df["reason_code"]
        == "NO_MAJOR_SIGNAL"
    ),
    "reason_code"
] = "LOW_CTR"


# ------------------------------------------------------------
# Confidence note
# ------------------------------------------------------------

output_df["confidence_note"] = (
    "Directional baseline based on recent "
    "observed position history."
)


output_df.loc[
    output_df["history_days"] < 7,
    "confidence_note"
] = (
    "Limited history; baseline may be unstable."
)


# ------------------------------------------------------------
# What could make it wrong
# ------------------------------------------------------------

output_df["could_be_wrong_if"] = (
    "Search position changes suddenly or "
    "recent history does not represent the next day."
)


output_df.loc[
    output_df["history_days"] < 7,
    "could_be_wrong_if"
] = (
    "Limited history may not represent "
    "normal page behaviour."
)


# ------------------------------------------------------------
# Final priority ranking
# ------------------------------------------------------------

output_df = (
    output_df
    .sort_values(
        [
            "action_score",
            "predicted_position_change"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


output_df["priority_rank"] = (
    np.arange(len(output_df)) + 1
)


# ------------------------------------------------------------
# Top-20
# ------------------------------------------------------------

top20 = (
    output_df
    .head(TOP_N)
    .copy()
)


# ------------------------------------------------------------
# Display Top-20
# ------------------------------------------------------------

display(
    top20[
        [
            "priority_rank",
            "content_hash_id",
            "latest_date",
            "latest_position",
            "baseline_predicted_position",
            "predicted_position_change",
            "predicted_position_change_pct",
            "action_score",
            "action_level",
            "reason_code",
            "confidence_note",
            "could_be_wrong_if"
        ]
    ]
)


print()

print(
    "Top-20 pages available:",
    len(top20)
)

KeyError: 'baseline_predicted_position'

## 4. Weak picks + leakage check
Some high-priority pages can be wrong.

Typical reasons include:

- very short history,
- high daily position volatility,
- an unusual one-day movement,
- missing measurements,
- a sudden external search change,
- a page-specific event that the baseline cannot observe.

The baseline should therefore be treated as a review mechanism, not as
ground truth.

The queue must use only information available before the prediction date.

The actual next-day position is not used to construct the baseline score.

No future target, intervention flag or product decision is used.

In [7]:
# ============================================================
# Final Output — Top-20 Action Queue
# ============================================================

final_top20 = (
    output_df[
        output_df["leakage_check_pass"]
    ]
    .sort_values(
        ["action_score", "predicted_position_change"],
        ascending=[False, False]
    )
    .head(20)
    .copy()
)

final_top20.insert(
    0,
    "priority_rank",
    range(1, len(final_top20) + 1)
)

display(
    final_top20[
        [
            "priority_rank",
            "client_hash_id",
            "content_hash_id",
            "latest_date",
            "prediction_date",
            "latest_position",
            "baseline_predicted_position",
            "predicted_position_change",
            "predicted_position_change_pct",
            "position_slope",
            "action_score",
            "action_level",
            "reason_code",
            "history_days"
        ]
    ]
)

print(f"\nFinal Top-20 queue created: {len(final_top20)} pages")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.